# SMLM PCF Pipeline: Plotting & Pooling

This notebook aggregates calculated PCFs across all processed nuclei, computes the pooled average with standard errors, and generates publication-quality plots.

### Steps:
1. **Aggregate**: Scan the output folder for all `*_nucleus_*_pcf.csv` files.
2. **Pool**: Compute mean ± SEM across nuclei for Auto-Left, Auto-Right, and Cross PCF.
3. **Plot**: 3-panel figure with individual curves (thin grey) + pooled mean ± SEM (thick coloured).

### Step 1: Configuration

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# PARAMETERS - edit these
# ==========================================
results_dir   = 'validation/test_dSTORM_data'
left_label    = 'SPEN (JF549)'       # label for left channel
right_label   = 'H3K27ac (DL650)'    # label for right channel

plot_output_path = os.path.join(results_dir, 'pooled_dstorm_pcf_curves.png')
pooled_csv_path  = os.path.join(results_dir, 'aggregated_pooled_pcf_results.csv')

print('Plotting parameters loaded successfully!')

### Step 2: Load and Pool Individual Nucleus CSVs

In [ ]:
csv_files = sorted(glob.glob(os.path.join(results_dir, '*_nucleus_*_pcf.csv')))
print(f'Found {len(csv_files)} nucleus PCF files:')

nucleus_data = []
for f in csv_files:
    df = pd.read_csv(f)
    filename = os.path.basename(f)
    parts    = filename.replace('_pcf.csv', '').split('_nucleus_')
    fov_name = parts[0]
    label_id = parts[1]
    nucleus_data.append({
        'fov':      fov_name,
        'label':    label_id,
        'r_center': df['r_center_nm'].values,
        'G_left':   df['G_auto_left'].values,
        'G_right':  df['G_auto_right'].values,
        'G_cross':  df['G_cross'].values,
    })
    print(f'  Loaded [FOV: {fov_name} | Nucleus: {label_id}]')

if len(nucleus_data) == 0:
    raise ValueError('No PCF files found! Run SMLM_PCF_Processing.ipynb first.')

# Build matrices and compute mean +/- SEM
r_centers  = nucleus_data[0]['r_center']
all_left   = np.array([d['G_left']  for d in nucleus_data])
all_right  = np.array([d['G_right'] for d in nucleus_data])
all_cross  = np.array([d['G_cross'] for d in nucleus_data])
n          = len(nucleus_data)

mean_left,  sem_left  = np.nanmean(all_left,  axis=0), np.nanstd(all_left,  axis=0) / np.sqrt(n)
mean_right, sem_right = np.nanmean(all_right, axis=0), np.nanstd(all_right, axis=0) / np.sqrt(n)
mean_cross, sem_cross = np.nanmean(all_cross, axis=0), np.nanstd(all_cross, axis=0) / np.sqrt(n)

pd.DataFrame({
    'r_center_nm':       r_centers,
    'mean_G_auto_left':  mean_left,  'sem_G_auto_left':  sem_left,
    'mean_G_auto_right': mean_right, 'sem_G_auto_right': sem_right,
    'mean_G_cross':      mean_cross, 'sem_G_cross':      sem_cross,
}).to_csv(pooled_csv_path, index=False)
print(f'\nPooled table saved to: {pooled_csv_path}')

### Step 3: Publication-Quality 3-Panel Figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)
r_min, r_max_plot = r_centers.min(), r_centers.max()

panel_cfg = [
    ('G_left',  all_left,  mean_left,  sem_left,  '#0288d1',
     f'Auto-Correlation\nLeft Channel ({left_label})'),
    ('G_right', all_right, mean_right, sem_right, '#d32f2f',
     f'Auto-Correlation\nRight Channel ({right_label})'),
    ('G_cross', all_cross, mean_cross, sem_cross, '#7b1fa2',
     f'Cross-Correlation\n({left_label} vs {right_label})'),
]

for ax, (gkey, all_g, mean_g, sem_g, color, title) in zip(axes, panel_cfg):
    # Individual nuclei - thin grey
    for d in nucleus_data:
        ax.plot(r_centers, d[gkey], color='#b0bec5', alpha=0.45, linewidth=1)
    # Pooled mean + SEM shading
    ax.plot(r_centers, mean_g, color=color, linewidth=3, label=f'Mean (n={n})')
    ax.fill_between(r_centers, mean_g - sem_g, mean_g + sem_g,
                    color=color, alpha=0.15, label='\u00b1SEM')
    ax.axhline(1.0, color='#263238', linestyle='--', alpha=0.6, linewidth=1.2)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Distance r (nm)', fontsize=11)
    ax.set_xlim(r_min, r_max_plot)
    ax.set_ylim(0.5, 2.0)
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(True, linestyle=':', alpha=0.5)

axes[0].set_ylabel('G(r)', fontsize=12)

plt.suptitle(
    f'Pooled SMLM PCF Analysis  (n={n} nuclei)\n'
    'Filters: area \u2265 6\u2009000 px  &  border margin \u2265 5 px',
    fontsize=13, fontweight='bold', y=1.04
)
plt.tight_layout()
plt.savefig(plot_output_path, dpi=300, bbox_inches='tight')
print(f'Plot saved to: {plot_output_path}')
plt.show()